# On-Demand LLM worker on a Kaggle GPU (tracker 4.6 / 4.8)

Runs the same llama.cpp CUDA build as `worker/Dockerfile.cuda`, directly in the notebook (Kaggle has no Docker),
records load time and tokens/sec, and exposes the server through a Cloudflare quick tunnel.

**Before running:** right panel -> *Session options* -> **Accelerator: GPU T4 x2 or P100**, **Internet: On**
(needs a phone-verified Kaggle account). Then *Run All*. Run cells in order.

The tunnel URL is public and has no auth: anyone with it can use your GPU session. Don't share it.

In [ ]:
# Settings (keep in sync with worker/Dockerfile.cuda)
LLAMA_RELEASE = "b11960"
CUDA_TAG = "cuda-12.8"
MODEL_REPO = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
MODEL_QUANT = "Q4_K_M"
CTX_SIZE = 8192
PORT = 8080
WORK = "/kaggle/working"

In [ ]:
# 1. Confirm a GPU is attached
!nvidia-smi

In [ ]:
%%bash -s "$LLAMA_RELEASE" "$CUDA_TAG" "$WORK"
set -e
REL=$1; CUDA=$2; WORK=$3
cd $WORK
BASE=https://github.com/ggml-org/llama.cpp/releases/download/$REL
curl -fsSL -o llama.tar.gz  $BASE/llama-$REL-bin-ubuntu-$CUDA-x64.tar.gz
curl -fsSL -o cudart.tar.gz $BASE/cudart-llama-$REL-bin-ubuntu-$CUDA-x64.tar.gz
mkdir -p llama/bin models
tar -xzf llama.tar.gz  -C llama/bin --strip-components=1
tar -xzf cudart.tar.gz -C llama/bin --strip-components=1
rm llama.tar.gz cudart.tar.gz
ls llama/bin | head -20

In [ ]:
# 2. Start llama-server and time how long until /health is 200 (task 4.8: GPU load time)
import os, subprocess, time, urllib.request, urllib.error

env = dict(os.environ, LD_LIBRARY_PATH=f"{WORK}/llama/bin", LLAMA_CACHE=f"{WORK}/models")
log = open(f"{WORK}/server.log", "w")
t0 = time.time()
server = subprocess.Popen(
    [f"{WORK}/llama/bin/llama-server", "-hf", f"{MODEL_REPO}:{MODEL_QUANT}",
     "--host", "0.0.0.0", "--port", str(PORT), "--ctx-size", str(CTX_SIZE), "--n-gpu-layers", "999"],
    env=env, stdout=log, stderr=subprocess.STDOUT)

def healthy():
    try:
        return urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=3).status == 200
    except (urllib.error.URLError, OSError):
        return False

while not healthy():
    if server.poll() is not None:
        raise RuntimeError("llama-server exited. Last log lines:\n" + open(f"{WORK}/server.log").read()[-3000:])
    if time.time() - t0 > 900:
        raise TimeoutError("not healthy after 15 min")
    time.sleep(2)

LOAD_SECONDS = time.time() - t0
print(f"healthy after {LOAD_SECONDS:.0f}s (includes the model download on a first run; re-run for the cached time)")

In [ ]:
# 3. Benchmark: tokens/sec (task 4.8)
import json
req = urllib.request.Request(
    f"http://127.0.0.1:{PORT}/v1/chat/completions",
    data=json.dumps({"messages": [{"role": "user", "content": "Write a short paragraph about the ocean."}],
                     "max_tokens": 200}).encode(),
    headers={"Content-Type": "application/json"})
t = time.time()
res = json.load(urllib.request.urlopen(req, timeout=300))
wall = time.time() - t
timings = res.get("timings", {})
print(res["choices"][0]["message"]["content"][:300], "...\n")
print(f"wall time:          {wall:.1f}s")
print(f"prompt eval:        {timings.get('prompt_per_second', float('nan')):.1f} tokens/s")
print(f"generation:         {timings.get('predicted_per_second', float('nan')):.1f} tokens/s")
print(f"model load/health:  {LOAD_SECONDS:.0f}s")

In [ ]:
# 4. Public tunnel (Cloudflare quick tunnel, no account needed)
import re
!curl -fsSL -o {WORK}/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x {WORK}/cloudflared

tlog = open(f"{WORK}/tunnel.log", "w")
tunnel = subprocess.Popen([f"{WORK}/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
                          stdout=tlog, stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(f"{WORK}/tunnel.log").read())
    if m:
        url = m.group(0)
        break
    time.sleep(1)
print("WORKER_URL =", url or "not found, check tunnel.log")
print("Put this in the gateway's .env as WORKER_URL, then check:", (url or "") + "/health")

In [ ]:
# 5. Keep the session alive while you test (stop this cell to end). Kaggle ends idle sessions.
while True:
    if server.poll() is not None:
        print("llama-server stopped")
        break
    time.sleep(30)